## Task: Build a Campus FAQ Chatbot using RAG

### Objective:
Learn how Retrieval-Augmented Generation (RAG) works by building a small chatbot that answers questions about your college using vector embeddings and a mini vector database.

#### Step 0: Setup

1. Install required packages:

In [ ]:
%pip install streamlit sentence-transformers faiss-cpu numpy

#### Step 1: Prepare the Data

Task: Create a small FAQ dataset with at least 5 Q&A pairs.
Example:

Q: When does the library open?
A: The library opens at 8 AM and closes at 8 PM.

In [1]:
# Create a small FAQ dataset for campus information
faq_text = """
Q: When does the library open?
A: The library opens at 8 AM and closes at 8 PM.

Q: Where is the student cafeteria located?
A: The student cafeteria is located on the second floor of the Student Union building.

Q: What are the office hours for academic advising?
A: Academic advising office hours are Monday through Friday, 9 AM to 5 PM.

Q: How do I access the campus WiFi?
A: Connect to "CampusWiFi" network and use your student ID and password to log in.

Q: Where can I find parking on campus?
A: Student parking is available in lots A, B, and C. Lot A is closest to the academic buildings.

Q: When is the registration deadline for next semester?
A: Registration deadline for next semester is November 15th for spring and April 15th for fall.
"""

print("FAQ dataset created with the following content:")
print(faq_text)

FAQ dataset created with the following content:

Q: When does the library open?
A: The library opens at 8 AM and closes at 8 PM.

Q: Where is the student cafeteria located?
A: The student cafeteria is located on the second floor of the Student Union building.

Q: What are the office hours for academic advising?
A: Academic advising office hours are Monday through Friday, 9 AM to 5 PM.

Q: How do I access the campus WiFi?
A: Connect to "CampusWiFi" network and use your student ID and password to log in.

Q: Where can I find parking on campus?
A: Student parking is available in lots A, B, and C. Lot A is closest to the academic buildings.

Q: When is the registration deadline for next semester?
A: Registration deadline for next semester is November 15th for spring and April 15th for fall.



Checkpoint:

Students should have a list of questions and answers ready.

#### Step 2: Split Text into Chunks

Task: Split your FAQ into separate lines to treat each Q&A as a chunk.

In [4]:
lines = [chunk.strip() for chunk in faq_text.split("\n\n") if chunk.strip()]
lines

['Q: When does the library open?\nA: The library opens at 8 AM and closes at 8 PM.',
 'Q: Where is the student cafeteria located?\nA: The student cafeteria is located on the second floor of the Student Union building.',
 'Q: What are the office hours for academic advising?\nA: Academic advising office hours are Monday through Friday, 9 AM to 5 PM.',
 'Q: How do I access the campus WiFi?\nA: Connect to "CampusWiFi" network and use your student ID and password to log in.',
 'Q: Where can I find parking on campus?\nA: Student parking is available in lots A, B, and C. Lot A is closest to the academic buildings.',
 'Q: When is the registration deadline for next semester?\nA: Registration deadline for next semester is November 15th for spring and April 15th for fall.']

Checkpoint:

Ensure each Q&A is a separate element in a Python list.

#### Step 3: Create Embeddings

Task: Convert each line to a vector using SentenceTransformer.

In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(lines)

d:\Lambton\OneDrive - Lambton College\AIMO Course\Semester 3\AML-3303 Software Tools and Emerging\ProjectFiles\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Lambton\OneDrive - Lambton College\AIMO Course\Semester 3\AML-3303 Software Tools and Emerging\ProjectFiles\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ryan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/hugg

#### Step 4: Build the FAISS Index

Task: Store all embeddings in a FAISS vector database.

In [6]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

#### Step 5: Query the Database
Task: Take a user question, convert it to a vector, and find the most relevant FAQ line.

In [14]:
user_question = "What time does the library open?"
q_emb = model.encode([user_question])
D, I = index.search(np.array(q_emb), k=1)
print("Answer:", lines[I[0][0]])

Answer: Q: When does the library open?
A: The library opens at 8 AM and closes at 8 PM.


#### Step 6: Make it Interactive with Streamlit
Task: Use Streamlit to create a simple chatbot UI.

In [15]:
import streamlit as st

st.title("Campus FAQ Chatbot")
user_question = st.text_input("Ask your question:")
if user_question:
    q_emb = model.encode([user_question])
    D, I = index.search(np.array(q_emb), k=1)
    st.write("Answer:", lines[I[0][0]])


2025-11-07 17:20:36.091 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-07 17:20:36.342 
  command:

    streamlit run d:\Lambton\OneDrive - Lambton College\AIMO Course\Semester 3\AML-3303 Software Tools and Emerging\ProjectFiles\venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-11-07 17:20:36.343 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-07 17:20:36.344 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-07 17:20:36.344 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-07 17:20:36.344 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-07 17:20:36.345 Thread 'MainThread': missing ScriptRunContext! This warnin

#### Step 7: Reflection

1. **How does the chatbot "understand" the question?**
    - The chatbot converts both the user's question and the FAQ entries into numerical vectors (embeddings) using the SentenceTransformer model
    - These embeddings capture the semantic meaning of the text in a high-dimensional space
    - When a question is asked, the system finds the FAQ entry with the most similar embedding using cosine similarity (via FAISS search)
    - The chatbot doesn't truly "understand" like humans do, but rather matches patterns in the mathematical representation of language

2. **What happens if the user asks something not in the FAQ?**
    - The system will still return the "closest" match from the existing FAQ entries, even if it's not relevant
    - There's no mechanism to detect when a question is outside the scope of the knowledge base
    - This leads to misleading or incorrect answers being provided to users

3. **How could you improve this system to handle more questions or longer documents?**
    - **Add confidence thresholds**: Only return answers when similarity scores are above a certain threshold
    - **Expand the knowledge base**: Include more comprehensive campus information and policies
    - **Implement chunking strategies**: For longer documents, split them into smaller, more focused chunks
    - **Add fallback responses**: When no good match is found, provide helpful alternatives like "I don't know, but you can contact..."
    - **Use more advanced models**: Implement larger language models or domain-specific embeddings
    - **Add query preprocessing**: Handle typos, synonyms, and different phrasings of the same question
    - **Implement feedback loops**: Allow users to rate answers and retrain the system accordingly